# AI应用基础 · 工单M4.1《工业故障报告自动分类系统》 — 教师参考版
第10周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m4"], check=True)
print("数据已生成: data/")

工作目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/01-AI应用基础
M4.1 -> data/fault_reports.csv 225 条
完成。数据输出目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/01-AI应用基础/data
数据已生成: data/


## 参考实现 `m41`（含依赖导入）

In [2]:
import csv
import json
import sys
import numpy as np
import numpy as np  # 通用科学计算


def m41():
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import classification_report
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.pipeline import make_pipeline
    import jieba

    texts, labels = [], []
    with open("data/fault_reports.csv", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            texts.append(r["text"])
            labels.append(r["label"])
    stop = set("的 了 在 是 请 尽快 处理 影响 生产 班中 巡检 发现 已 挂牌 停机".split())

    def cut(t):
        return " ".join(w for w in jieba.cut(t) if w not in stop and w.strip())

    corpus = [cut(t) for t in texts]
    n = int(len(corpus) * 0.8)
    for name, clf in [("NB", MultinomialNB()),
                      ("LR", LogisticRegression(max_iter=1000))]:
        pipe = make_pipeline(TfidfVectorizer(max_features=2000), clf)
        pipe.fit(corpus[:n], labels[:n])
        pred = pipe.predict(corpus[n:])
        print("=====", name, "=====")
        print(classification_report(labels[n:], pred))
    demo = "3号注塑机液压油温升高报警"
    pipe2 = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
    pipe2.fit(corpus, labels)
    print(demo, "->", pipe2.predict([cut(demo)]))


## 学生版TODO → 教师答案对照
学生版starter中的TODO与上方参考实现（或下方双解）的对应关系：

| TODO | 题面（学生看到的指导） |
|------|----------------------|
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） |
| `2` | TODO-2: fit(tr_x, tr_y) → predict(te_x) → 记录accuracy |
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） |
| `3` | TODO-3: 新文本预测(自收集的2条+示例) |
| `4` | TODO-4: 每类Top10关键词(pipe[:-1].get_feature_names_out + |
|  | coef/feature_log_prob_) TODO-5: (优秀)混淆矩阵: 哪两类易混? 为什么? |

> 发放纪律：本对照表**勿随学生版下发**；课堂按层放提示（D层演示照做/C层只给思路行）。

## 执行与验证

In [3]:
m41()

Building prefix dict from the default dictionary ...


Loading model from cache /var/folders/6j/khvhkzp55m30fx539sm9tggr0000gn/T/jieba.cache


Loading model cost 0.375 seconds.


Prefix dict has been built successfully.


===== NB =====
              precision    recall  f1-score   support

        仪表气动       1.00      1.00      1.00        10
        传动系统       1.00      1.00      1.00         7
        冷却系统       1.00      1.00      1.00        10
        液压系统       1.00      1.00      1.00        11
        电气系统       1.00      1.00      1.00         7

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45

===== LR =====
              precision    recall  f1-score   support

        仪表气动       1.00      1.00      1.00        10
        传动系统       1.00      1.00      1.00         7
        冷却系统       1.00      1.00      1.00        10
        液压系统       1.00      1.00      1.00        11
        电气系统       1.00      1.00      1.00         7

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00    